# Challenge 2: Online Purchase Prediction (final notebook)

**Team 5, CAI 4105 Fall 2026:** Daniel Ibarra, Joaquin De La Barrera, Jiayang Li, Kevin Herrera

This notebook shows the complete workflow, from loading the data to writing `predictions.csv`:
1. Data exploration and preprocessing (shared code in `helpers.py`; full EDA in `step1-2_base.ipynb`)
2. Validation strategy: stratified 5-fold CV, F1 for `Revenue=True`
3. and 4. Comparison of all 7 required models after Grid Search (full runs in `step3_linear.ipynb` and `step3_trees.ipynb`)
5. Final model selection, re-validated here
6. Refit on all training data and predict the test file

Run from the `Challenge 2/` folder after `pip install -r requirements.txt`, then Restart & Run All. No test labels are used or inferred.

## Step 1: Data exploration and preprocessing

`load_data()` returns the 17 feature columns (no `ID`) and `Revenue` as 0/1. Preprocessing (in `helpers.py`, always inside a `Pipeline` with the model so each CV fold fits it on its own training part):
- 10 numeric columns: median imputation, plus `StandardScaler` for scale-sensitive models (LogReg, KNN, SVM, Naive Bayes). The linear-model notebook also applies `log1p` to the skewed counts, durations and `PageValues`, which raised CV F1 by about 0.12 for LogReg and KNN.
- 7 categorical columns (`Month`, `VisitorType`, `Weekend`, and the code columns `OperatingSystems`, `Browser`, `Region`, `TrafficType`): most-frequent imputation and one-hot encoding with `handle_unknown="ignore"`.

In [1]:
import pandas as pd
from helpers import NUM_COLS, CAT_COLS, load_data as _load
_X, _y = _load()
print('Shape:', _X.shape, '| missing values:', int(_X.isna().sum().sum()))
print('Target balance:', _y.value_counts(normalize=True).round(3).to_dict(), '(1 = purchase)')
print('Buy rate when PageValues == 0:', round(_y[_X['PageValues'] == 0].mean(), 3),
      '| when PageValues > 0:', round(_y[_X['PageValues'] > 0].mean(), 3))
print('Buy rate by VisitorType:', _y.groupby(_X['VisitorType']).mean().round(3).to_dict())
_X[NUM_COLS].describe().T.round(2)

Shape: (9247, 17) | missing values: 0
Target balance: {0: 0.845, 1: 0.155} (1 = purchase)
Buy rate when PageValues == 0: 0.036 | when PageValues > 0: 0.569
Buy rate by VisitorType: {'New_Visitor': 0.247, 'Other': 0.181, 'Returning_Visitor': 0.14}


,count,mean,std,min,25%,50%,75%,max
Administrative,9247.0,2.31,3.32,0.0,0.00,1.00,4.00,27.00
Administrative_Duration,9247.0,81.17,179.59,0.0,0.00,8.00,92.55,3398.75
Informational,9247.0,0.50,1.27,0.0,0.00,0.00,0.00,24.00
Informational_Duration,9247.0,34.52,141.31,0.0,0.00,0.00,0.00,2549.38
ProductRelated,9247.0,31.36,43.54,0.0,7.00,18.00,38.00,705.00
ProductRelated_Duration,9247.0,1181.85,1908.10,0.0,181.42,598.25,1464.99,63973.52
BounceRates,9247.0,0.02,0.05,0.0,0.00,0.00,0.02,0.20
ExitRates,9247.0,0.04,0.05,0.0,0.01,0.03,0.05,0.20
PageValues,9247.0,5.98,18.75,0.0,0.00,0.00,0.00,361.76
SpecialDay,9247.0,0.06,0.20,0.0,0.00,0.00,0.00,1.00


## Step 2: Validation strategy

Because only about 15.5% of sessions buy, every model uses the same **stratified 5-fold CV** (`StratifiedKFold(5, shuffle=True, random_state=42)`, `helpers.CV`), so each fold keeps the buyer rate. The primary metric is **F1 for `Revenue=True`**. We also report precision, recall, accuracy and the out-of-fold confusion matrix, and compare train F1 with CV F1 to detect overfitting. Grid Search uses the same folds and F1 scoring. Baselines: always predicting "no purchase" gives F1 = 0 (84.5% accuracy); a stratified random guess gives CV F1 = 0.141.

## Steps 3 and 4: Model comparison after Grid Search

Six of the seven required models were tuned with `GridSearchCV` on the training data only. The table reproduces the measured outputs of `step3_linear.ipynb` (Joaquin) and `step3_trees.ipynb` (Jiayang). Decision Tree was kept at its default because Random Forest and Gradient Boosting (tree ensembles) dominate it.

In [2]:
comparison = pd.DataFrame([
    ('Random Forest (tuned)',       0.785, 0.682, 0.008, 'class_weight=balanced, max_depth=None, min_samples_leaf=5', 'step3_trees'),
    ('Logistic Regression (tuned)', 0.679, 0.680, 0.009, 'C=0.03, L1, class_weight=balanced, log1p features', 'step3_linear'),
    ('SVM RBF (tuned)',             0.678, 0.677, 0.010, 'C=1, gamma=0.003, class_weight=balanced, log1p features', 'step3_linear'),
    ('Gradient Boosting (tuned)',   0.720, 0.670, 0.020, 'n_estimators=150, learning_rate=0.05, max_depth=3', 'step3_trees'),
    ('KNN (tuned)',                 0.647, 0.633, 0.022, 'k=31, uniform, p=2, log1p features', 'step3_linear'),
    ('Naive Bayes (tuned)',         0.602, 0.600, 0.011, 'var_smoothing=1.0, log1p features', 'step3_linear'),
    ('Decision Tree (default)',     1.000, 0.563, 0.024, 'default (overfits: train F1 = 1.0)', 'step3_trees'),
    ('Baseline: stratified guess',  0.157, 0.141, 0.015, 'DummyClassifier', 'step3_linear'),
], columns=['model', 'train_f1', 'cv_f1', 'cv_f1_std', 'best settings', 'source notebook']).set_index('model')
comparison

,train_f1,cv_f1,cv_f1_std,best settings,source notebook
model,,,,,
Random Forest (tuned),0.785,0.682,0.008,"class_weight=balanced, max_depth=None, min_sam...",step3_trees
Logistic Regression (tuned),0.679,0.680,0.009,"C=0.03, L1, class_weight=balanced, log1p features",step3_linear
SVM RBF (tuned),0.678,0.677,0.010,"C=1, gamma=0.003, class_weight=balanced, log1p...",step3_linear
Gradient Boosting (tuned),0.720,0.670,0.020,"n_estimators=150, learning_rate=0.05, max_depth=3",step3_trees
KNN (tuned),0.647,0.633,0.022,"k=31, uniform, p=2, log1p features",step3_linear
Naive Bayes (tuned),0.602,0.600,0.011,"var_smoothing=1.0, log1p features",step3_linear
Decision Tree (default),1.000,0.563,0.024,default (overfits: train F1 = 1.0),step3_trees
Baseline: stratified guess,0.157,0.141,0.015,DummyClassifier,step3_linear


## Step 5: Select the final model

**Final choice: tuned Random Forest** (`class_weight='balanced'`, `min_samples_leaf=5`, 100 trees).
- **Validation performance:** highest CV F1 of all models (0.682), re-measured below on the shared pipeline. Logistic Regression (0.680) and SVM (0.677) are within one standard deviation, so they are close alternatives.
- **Overfitting:** the default forest overfit badly (train F1 1.000 vs CV 0.617); `min_samples_leaf=5` shrinks the gap to about 0.10, and its CV std (0.008) is the lowest of all models, so it is the most stable across folds.
- **Complexity and cost:** less interpretable and slower than Logistic Regression, but it fits in about 2 seconds, which is cheap here. SVM was the slowest to tune (324 s).
- **Interpretability:** Logistic Regression would be easier to explain; we accept that trade-off for the best and most stable F1.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from helpers import CV, SCORING, RANDOM_STATE, load_data, load_test, make_model_pipeline

X, y = load_data()
assert len(X) == 9247 and len(y) == 9247
assert 'ID' not in X.columns and 'Revenue' not in X.columns
print('scikit-learn:', sklearn.__version__)
print('Training data:', X.shape, 'buyers:', int(y.sum()))

scikit-learn: 1.5.2
Training data: (9247, 17) buyers: 1431


### Final model configuration

The parameters below are the best `GridSearchCV` parameters from `step3_trees.ipynb` (`class_weight='balanced'`, `max_depth=None`, `min_samples_leaf=5`; other settings default, `random_state=42`). The CV metrics printed below are this notebook's own measurement.

In [4]:
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
pipeline = make_model_pipeline(final_model, scale=False)
cv_result = cross_validate(pipeline, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=1)
metrics = {k: float(np.mean(v)) for k, v in cv_result.items() if k.startswith('test_') or k.startswith('train_')}
print('CV F1:', round(metrics['test_f1'], 4), '+/-', round(float(np.std(cv_result['test_f1'])), 4))
print('Train F1:', round(metrics['train_f1'], 4))
print('CV precision:', round(metrics['test_precision'], 4), 'recall:', round(metrics['test_recall'], 4), 'accuracy:', round(metrics['test_accuracy'], 4))
print('Train-CV F1 gap:', round(metrics['train_f1'] - metrics['test_f1'], 4))
cv_predictions = cross_val_predict(pipeline, X, y, cv=CV, n_jobs=1)
print('Out-of-fold confusion matrix:')
print(confusion_matrix(y, cv_predictions))
print(classification_report(y, cv_predictions, zero_division=0))

CV F1: 0.6818 +/- 0.0081
Train F1: 0.7847
CV precision: 0.6034 recall: 0.7847 accuracy: 0.8867
Train-CV F1 gap: 0.1028


Out-of-fold confusion matrix:
[[7076  740]
 [ 308 1123]]
              precision    recall  f1-score   support

           0       0.96      0.91      0.93      7816
           1       0.60      0.78      0.68      1431

    accuracy                           0.89      9247
   macro avg       0.78      0.85      0.81      9247
weighted avg       0.90      0.89      0.89      9247



## Step 6 — Refit and export

The test data is loaded **only after model selection and cross-validation**. The `ID` column is retained for the submission, never used as a feature. Output is written beside this notebook as `Challenge 2/predictions.csv`.

In [5]:
X_test, test_ids = load_test()
assert len(X_test) == 3083 and len(test_ids) == 3083
assert X_test.columns.tolist() == X.columns.tolist()
assert test_ids.is_unique and not test_ids.isna().any()
pipeline.fit(X, y)
predicted = pipeline.predict(X_test).astype(bool)
submission = pd.DataFrame({'ID': test_ids, 'Revenue': predicted})
assert submission.columns.tolist() == ['ID', 'Revenue']
assert len(submission) == 3083 and submission['Revenue'].notna().all()
assert submission['Revenue'].isin([True, False]).all()
output_path = Path('predictions.csv')
submission.to_csv(output_path, index=False)
check = pd.read_csv(output_path)
assert len(check) == 3083 and check.columns.tolist() == ['ID', 'Revenue']
assert check['ID'].equals(test_ids.reset_index(drop=True))
print('Saved:', output_path.resolve())
print('Rows:', len(check), '| predicted purchases:', int(predicted.sum()))
print(check.head())

Saved:

 C:\Users\danni\Documents\Programming\Projects\CAI_4105_2026_Fall_Team_5\Challenge 2\predictions.csv
Rows: 3083 | predicted purchases: 615
   ID  Revenue
0   2    False
1   5    False
2   6    False
3   8    False
4  10    False


## Reproducibility

- Environment: Python 3.11 with the pinned packages in `requirements.txt`; `random_state=42` everywhere.
- From `Challenge 2/`: `pip install -r requirements.txt`, then Restart & Run All (about 15 seconds). It rewrites `predictions.csv` with identical predictions.
- The test file is loaded only in Step 6, and `ID` is never used as a feature.